# 00C Demo Setup — reset and prepare Day 1

Load the existing `00_env_config`, run the single reset cell, then explore the I/O examples. Start with `RESET_DEMO = False` to preview. Set it to `True` only for the dedicated, manually created `demo_metadata` lakehouse and reserved `demo` business tables.


## 1. Load the shared Fabric configuration

The setup notebook uses the same workspace-local `00_env_config` as the rest of the Guided Demo.


In [1]:
from fabricops_kit import setup_notebook

### Use this when the 00_env_config notebook is in the same workspace as this notebook
context = setup_notebook()

### Advanced you can run the 00_env_config notebook that is in another workspace, 
### make sure you have workspace execution rights to that workspace for this to work
### The caller workspace is detected automatically. Configuration workspace example: "<configuration-workspace-id>"
# Advanced: load 00_env_config from another workspace:
# context = setup_notebook(
#     config_workspace_id="<configuration-workspace-id>",
# )


## 2. Import the FabricOps I/O helpers


In [ ]:
from fabricops_kit import (
    read_lakehouse_csv,
    read_lakehouse_excel,
    read_lakehouse_json,
    read_lakehouse_parquet,
    read_lakehouse_table,
    read_warehouse_query,
    read_warehouse_table,
    write_lakehouse_table,
    write_warehouse_table,
)


## Reset first, then explore the I/O examples

Stop all demo jobs. `False` previews; `True` clears the dedicated `demo_metadata` tables and saved lifecycle artifacts, removes six outputs and restores five Day 1 sources. Contracts, activations and Run history are discarded. Use only the manually created demo item selected by `00_env_config`. The cell checks non-demo references and active jobs before deletion. Wait for `RESET COMPLETE`.


In [ ]:
RESET_DEMO = False  # False: preview only. True: reset the dedicated demo environment.

from fabricops_kit import orchestrate_demo_reset

orchestrate_demo_reset(reset=RESET_DEMO)


## 3. Read the same "Orders" dataset via CSV, JSON, Parquet, and Excel.

All four files contain the same 120 logical Orders rows. Paths are relative to the configured Lakehouse `Files` area, so `demo/orders.csv` resolves under `Files/demo/orders.csv`.


In [ ]:
orders_csv_df = read_lakehouse_csv(
    "demo/orders.csv",
    store="Bronze",
    
    header=True,
    inferSchema=True,
)

#display(orders_csv_df)


In [ ]:
orders_json_df = read_lakehouse_json(
    "demo/orders.json",
    store="Bronze",
    
)
#display(orders_json_df)


In [ ]:
orders_parquet_df = read_lakehouse_parquet(
    "demo/orders.parquet",
    store="Bronze",
    
)

#display(orders_parquet_df)


In [ ]:
orders_excel_df = read_lakehouse_excel(
    "demo/orders.xlsx",
    store="Bronze",
    
)
#display(orders_excel_df)


## 4. Read the restored Lakehouse and Warehouse sources

The reset cell wrote these tables with the public write helpers. The following cells demonstrate the public readers.


In [ ]:
# Sources were written and verified in the reset cell. Inspect with the public reader.
if RESET_DEMO:
    orders_table_df = read_lakehouse_table(table_name="orders", store="Bronze", schema="demo")


In [ ]:
if RESET_DEMO:
    products_table_df = read_lakehouse_table(table_name="products", store="Bronze", schema="demo")


In [ ]:
if RESET_DEMO:
    order_history_table_df = read_warehouse_table(table_name="order_history", store="Gold", schema="demo")
    order_history_sample_df = read_warehouse_query("SELECT TOP 5 * FROM demo.order_history ORDER BY 1", store="Gold")


## 5. Inspect Day 1 append and SCD sources

The reset cell already restored both sources. Do not reset between scenario days.


In [ ]:
if RESET_DEMO:
    for table in ("inventory_movements", "product_master_updates"):
        print(f"Bronze demo.{table}:", read_lakehouse_table(table, store="Bronze", schema="demo").count(), "rows")
